In [1]:
!git clone https://github.com/SarhadGautam/AI_Steel_Hackathon_Sarhad.git

Cloning into 'AI_Steel_Hackathon_Sarhad'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 27 (delta 8), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 671.62 KiB | 4.22 MiB/s, done.
Resolving deltas: 100% (8/8), done.


In [2]:
%cd AI_Steel_Hackathon_Sarhad
%cd dataset

/content/AI_Steel_Hackathon_Sarhad
/content/AI_Steel_Hackathon_Sarhad/dataset


In [31]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import *


#print(os.getcwd())
print(os.listdir())
pd.set_option(
    'display.float_format',
    '{:.2f}'.format
)

['sample_submission.csv', 'test.csv', 'submission.csv', 'train.csv']


In [4]:
train_df = pd.read_csv('train.csv', index_col = 'CoilID')
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1352 entries, 487 to 465
Data columns (total 50 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X1      1352 non-null   float64
 1   X2      1352 non-null   float64
 2   X3      1352 non-null   float64
 3   X4      1352 non-null   float64
 4   X5      1352 non-null   float64
 5   X6      1352 non-null   float64
 6   X7      1352 non-null   float64
 7   X8      1351 non-null   float64
 8   X9      1352 non-null   float64
 9   X10     1346 non-null   float64
 10  X11     1352 non-null   float64
 11  X12     1352 non-null   float64
 12  X13     1352 non-null   float64
 13  X14     1352 non-null   float64
 14  X15     1192 non-null   float64
 15  X16     1346 non-null   float64
 16  X17     1352 non-null   float64
 17  X18     1352 non-null   float64
 18  X19     1352 non-null   float64
 19  X20     1352 non-null   float64
 20  X21     1351 non-null   float64
 21  X22     1352 non-null   float64
 22  X23 

In [5]:
train_df['Y'].value_counts(normalize=True)*100

,proportion
Y,
0.00,95.12
1.00,4.88


In [6]:
#train_df.agg(['min', 'max', 'count', 'sum', 'median', ('mode', lambda x: x.mode().iloc[0])]).T -- doesnt work
#train_df.agg(['min', 'max', 'count', 'sum', 'median']).T.assign(mode=train_df.mode().iloc[0])
#train_df.agg(['min', 'max', 'count', 'sum', 'median']).T.assign(mode=train_df.apply(lambda x: ', '.join(map(str, round(x.mode(),2)))))
# mode above doesnt yield in any meaningful inference so dropping it altogether.
data_summary = train_df.agg(['min', 'max', 'count', 'sum', 'median', 'std']).T
data_summary['q25'] = train_df.quantile(.25)
data_summary['q50'] = train_df.quantile(.50)
data_summary['q75'] = train_df.quantile(.75)
data_summary['skeww'] = train_df.skew()
data_summary

,min,max,count,sum,median,std,q25,q50,q75,skeww
X1,235.25,1124.90,1352.00,1391093.52,1071.98,108.51,1009.28,1071.98,1092.03,-2.71
X2,96.76,1148.17,1352.00,777906.40,589.16,232.10,405.53,589.16,725.77,0.04
X3,124.15,1026.92,1352.00,727880.01,548.04,135.29,441.59,548.04,632.24,-0.23
X4,575.92,755.98,1352.00,936364.02,724.97,56.80,622.21,724.97,734.45,-0.80
X5,559.27,763.26,1352.00,877462.65,661.17,35.40,625.33,661.17,668.05,-0.27
X6,529.94,742.73,1352.00,836326.34,615.24,45.60,583.36,615.24,659.56,-0.11
X7,439.22,618.95,1352.00,715533.91,547.65,46.26,476.81,547.65,554.58,-0.51
X8,425.41,575.31,1351.00,714368.34,545.40,40.22,520.19,545.40,556.33,-1.08
X9,343.11,505.35,1352.00,624218.22,482.41,40.60,423.60,482.41,488.71,-1.19
X10,1.12,12.31,1346.00,9246.74,7.00,2.63,4.70,7.00,8.97,-0.10


In [7]:
null_summary = pd.DataFrame({
    'null_count': train_df.isnull().sum(),
    'null_pct': (train_df.isnull().mean()*100)
})
null_summary[null_summary['null_pct'] > 0.00].index

# Number of missing rows are less compared to the number of records here I will take the columns too with missing rows


Index(['X8', 'X10', 'X15', 'X16', 'X21', 'X23', 'X24', 'X25', 'X26', 'X27',
       'X42', 'X48'],
      dtype='object')

In [8]:
# Returns rows with missing data in those columns
null_columns = ['X8', 'X10', 'X15', 'X16', 'X21', 'X23', 'X24', 'X25', 'X26', 'X27',
       'X42', 'X48']
null_records = train_df[train_df[null_columns].isnull().any(axis=1)]
train_df.isnull().sum(axis=1).describe()


,0
count,1352.00
mean,0.18
std,0.62
min,0.00
25%,0.00
50%,0.00
75%,0.00
max,8.00


In [9]:
train_df.isnull().sum(axis=1).value_counts().sort_index()

,count
0,1159
1,172
2,12
3,3
7,4
8,2


In [11]:
null_records_df = train_df.loc[train_df[null_columns].isnull().any(axis=1),null_columns].sort_index()
null_records_df

,X8,X10,X15,X16,X21,X23,X24,X25,X26,X27,X42,X48
CoilID,,,,,,,,,,,,
745,537.73,6.42,3.03,17.06,636.65,24.14,18.00,12.69,13.36,10.58,NaN,NaN
750,516.67,6.36,3.12,20.17,698.56,24.77,16.49,14.25,13.65,9.56,NaN,0.00
752,479.32,4.44,4.95,16.18,689.91,23.18,11.37,12.29,10.93,8.15,NaN,NaN
753,472.78,4.71,3.92,16.83,660.26,24.64,12.11,12.51,11.52,8.90,NaN,NaN
754,485.77,4.82,4.02,16.58,663.91,24.38,12.23,12.08,11.34,8.55,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1625,441.36,2.92,NaN,29.10,722.52,34.13,-1.26,18.53,15.61,11.17,0.02,0.04
1626,437.82,1.84,NaN,28.45,854.40,34.45,-0.22,19.24,16.05,11.38,0.02,0.04
1628,440.50,2.28,NaN,30.40,743.04,34.49,0.42,18.56,15.66,11.24,0.02,0.04


In [12]:
pd.crosstab(
    train_df['X8'].isnull(),
    train_df['Y'],
    normalize='columns'
)

Y,0.00,1.00
X8,,
False,1.00,1.00
True,0.00,0.00


In [13]:
#train_df['X8'] = (train_df['X8'].fillna(train_df['X8'].median()))
for c in null_columns:
  train_df[c] = (train_df[c].fillna(train_df[c].median()))

In [14]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1352 entries, 487 to 465
Data columns (total 50 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X1      1352 non-null   float64
 1   X2      1352 non-null   float64
 2   X3      1352 non-null   float64
 3   X4      1352 non-null   float64
 4   X5      1352 non-null   float64
 5   X6      1352 non-null   float64
 6   X7      1352 non-null   float64
 7   X8      1352 non-null   float64
 8   X9      1352 non-null   float64
 9   X10     1352 non-null   float64
 10  X11     1352 non-null   float64
 11  X12     1352 non-null   float64
 12  X13     1352 non-null   float64
 13  X14     1352 non-null   float64
 14  X15     1352 non-null   float64
 15  X16     1352 non-null   float64
 16  X17     1352 non-null   float64
 17  X18     1352 non-null   float64
 18  X19     1352 non-null   float64
 19  X20     1352 non-null   float64
 20  X21     1352 non-null   float64
 21  X22     1352 non-null   float64
 22  X23 

In [15]:
cols = train_df.columns.to_list()
cols.remove('Y')
print(cols)

['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10', 'X11', 'X12', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19', 'X20', 'X21', 'X22', 'X23', 'X24', 'X25', 'X26', 'X27', 'X28', 'X29', 'X30', 'X31', 'X32', 'X33', 'X34', 'X35', 'X36', 'X37', 'X38', 'X39', 'X40', 'X41', 'X42', 'X43', 'X44', 'X45', 'X46', 'X47', 'X48', 'X49']


In [16]:
train_df.corr(numeric_only=True)['Y'].sort_values(key=abs,ascending=False)

,Y
Y,1.00
X35,-0.26
X13,0.25
X36,-0.24
X34,-0.24
X10,0.24
X30,0.22
X31,0.21
X32,0.21
X29,0.19


In [17]:
train_df[cols].corr(numeric_only = True).abs()

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X40,X41,X42,X43,X44,X45,X46,X47,X48,X49
X1,1.00,0.17,0.00,0.19,0.11,0.07,0.16,0.12,0.16,0.23,...,0.07,0.11,0.04,0.21,0.08,0.18,0.01,0.05,0.04,0.21
X2,0.17,1.00,0.48,0.27,0.17,0.26,0.26,0.36,0.34,0.39,...,0.16,0.28,0.28,0.02,0.08,0.11,0.03,0.24,0.11,0.29
X3,0.00,0.48,1.00,0.09,0.05,0.14,0.06,0.11,0.05,0.13,...,0.13,0.05,0.04,0.10,0.10,0.04,0.08,0.11,0.22,0.13
X4,0.19,0.27,0.09,1.00,0.80,0.73,0.89,0.78,0.84,0.75,...,0.43,0.38,0.49,0.20,0.17,0.16,0.15,0.00,0.12,0.17
X5,0.11,0.17,0.05,0.80,1.00,0.87,0.84,0.71,0.74,0.60,...,0.25,0.33,0.41,0.03,0.09,0.10,0.15,0.12,0.09,0.15
X6,0.07,0.26,0.14,0.73,0.87,1.00,0.80,0.71,0.69,0.60,...,0.09,0.34,0.37,0.01,0.11,0.22,0.20,0.08,0.12,0.18
X7,0.16,0.26,0.06,0.89,0.84,0.80,1.00,0.75,0.81,0.69,...,0.38,0.36,0.47,0.08,0.12,0.12,0.15,0.09,0.10,0.15
X8,0.12,0.36,0.11,0.78,0.71,0.71,0.75,1.00,0.88,0.70,...,0.32,0.40,0.48,0.16,0.17,0.25,0.19,0.03,0.08,0.25
X9,0.16,0.34,0.05,0.84,0.74,0.69,0.81,0.88,1.00,0.75,...,0.42,0.42,0.48,0.19,0.15,0.27,0.16,0.02,0.06,0.18
X10,0.23,0.39,0.13,0.75,0.60,0.60,0.69,0.70,0.75,1.00,...,0.44,0.46,0.51,0.33,0.21,0.29,0.13,0.08,0.17,0.27


In [18]:

X = train_df[cols]
y = train_df['Y']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [19]:
y_train.value_counts(normalize=True), y_test.value_counts(normalize=True)

(Y
 0.00   0.95
 1.00   0.05
 Name: proportion, dtype: float64,
 Y
 0.00   0.95
 1.00   0.05
 Name: proportion, dtype: float64)

In [20]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

lr_prob = lr.predict_proba(X_test)[:,1]

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [21]:
print("LOGISTIC REGRESSION")

print(classification_report(y_test, lr_pred))

print(confusion_matrix(y_test, lr_pred))

print("Precision:", precision_score(y_test, lr_pred))

print("Recall:", recall_score(y_test, lr_pred))

print("F1:", f1_score(y_test, lr_pred))

print("ROC AUC:", roc_auc_score(y_test, lr_prob))

print("PR AUC:", average_precision_score(y_test, lr_prob))

LOGISTIC REGRESSION
              precision    recall  f1-score   support

         0.0       0.99      0.73      0.84       258
         1.0       0.15      0.92      0.26        13

    accuracy                           0.74       271
   macro avg       0.57      0.83      0.55       271
weighted avg       0.95      0.74      0.82       271

[[189  69]
 [  1  12]]
Precision: 0.14814814814814814
Recall: 0.9230769230769231
F1: 0.2553191489361702
ROC AUC: 0.9025044722719142
PR AUC: 0.31358200408123005


In [22]:
rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

rf_prob = rf.predict_proba(X_test)[:,1]

In [23]:
print("RANDOM FOREST")

print(classification_report(y_test, rf_pred))

print("Confusion Matrix:", confusion_matrix(y_test, rf_pred))

print("Precision:", precision_score(y_test, rf_pred))

print("Recall:", recall_score(y_test, rf_pred))

print("F1:", f1_score(y_test, rf_pred))

print("ROC AUC:", roc_auc_score(y_test, rf_prob))

print("PR AUC:", average_precision_score(y_test, rf_prob))

RANDOM FOREST
              precision    recall  f1-score   support

         0.0       0.96      1.00      0.98       258
         1.0       1.00      0.08      0.14        13

    accuracy                           0.96       271
   macro avg       0.98      0.54      0.56       271
weighted avg       0.96      0.96      0.94       271

Confusion Matrix: [[258   0]
 [ 12   1]]
Precision: 1.0
Recall: 0.07692307692307693
F1: 0.14285714285714285
ROC AUC: 0.8765652951699464
PR AUC: 0.3462589340730124


In [24]:
gbm = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)

gbm.fit(X_train, y_train)

gbm_pred = gbm.predict(X_test)

gbm_prob = gbm.predict_proba(X_test)[:,1]

In [25]:
print("GRADIENT BOOSTING")

print(classification_report(y_test, gbm_pred))

print("Confusion Matrix", confusion_matrix(y_test, gbm_pred))

print("Precision:", precision_score(y_test, gbm_pred))

print("Recall:", recall_score(y_test, gbm_pred))



GRADIENT BOOSTING
              precision    recall  f1-score   support

         0.0       0.95      1.00      0.98       258
         1.0       0.00      0.00      0.00        13

    accuracy                           0.95       271
   macro avg       0.48      0.50      0.49       271
weighted avg       0.91      0.95      0.93       271

Confusion Matrix [[258   0]
 [ 13   0]]
Precision: 0.0
Recall: 0.0


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

In [26]:
test_df = pd.read_csv('test.csv', index_col = 'CoilID')
test_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 339 entries, 711 to 14
Data columns (total 49 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X1      339 non-null    float64
 1   X2      339 non-null    float64
 2   X3      339 non-null    float64
 3   X4      339 non-null    float64
 4   X5      339 non-null    float64
 5   X6      339 non-null    float64
 6   X7      339 non-null    float64
 7   X8      339 non-null    float64
 8   X9      339 non-null    float64
 9   X10     339 non-null    float64
 10  X11     339 non-null    float64
 11  X12     339 non-null    float64
 12  X13     339 non-null    float64
 13  X14     339 non-null    float64
 14  X15     287 non-null    float64
 15  X16     339 non-null    float64
 16  X17     339 non-null    float64
 17  X18     339 non-null    float64
 18  X19     339 non-null    float64
 19  X20     339 non-null    float64
 20  X21     339 non-null    float64
 21  X22     339 non-null    float64
 22  X23   

In [27]:
null_summary = pd.DataFrame({
    'null_count': test_df.isnull().sum(),
    'null_pct': (test_df.isnull().mean()*100)
})
null_summary[null_summary['null_pct'] > 0.00].index

Index(['X15', 'X42', 'X48'], dtype='object')

In [28]:
null_columns = ['X15', 'X42', 'X48']
for c in null_columns:
  test_df[c] = (test_df[c].fillna(test_df[c].median()))

In [29]:
cols = test_df.columns
test_df['Y'] = lr.predict(test_df[cols])


In [30]:
test_df['Y'].to_csv('submission.csv') # submission File getting stored in dataset folder